# 02 · Clean, label and split
Removes tiny files and duplicates, adds genus, family, WHO category and tier, and splits by photographer.

**Reads:** the raw dataset (01). **Saves:** `<your username>/serpentguard-m2-splits` with `splits.csv`, `classes.csv`, `excluded.csv` and `split_report.json`. **Time:** about 3 minutes on CPU.

`StratifiedGroupKFold` assigns folds differently between scikit-learn releases, so this notebook pins the version that made our split, and the last cell checks the result against the published file.

In [ ]:
!pip install -q "scikit-learn==1.9.1"

In [ ]:
import gzip, json, re, time
import urllib.error, urllib.request
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

RAW_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes"
SPLITS_HANDLE = "puneetbajajiitm/serpentguard-m2-splits"
UPLOAD = True            # save the result as a private dataset under your account

PREP_DIR = Path("/kaggle/tmp/serpentguard/prep")
PREP_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "SerpentGuard-course-project (github.com/Puneet-Bajaj-IITM/Group-5-DS-and-AI-Lab-Project)"
INAT_API = "https://api.inaturalist.org/v1"
TAXA_BATCH = 30
API_PAUSE_S = 1.1
RETRIES = 5
TIMEOUT_S = 90
MIN_BYTES = 5_000
MIN_SIDE_PX = 100
MIN_PHOTOS_PER_CLASS = 20
TIER1_MIN_PHOTOS = 400
N_FOLDS = 7
TEST_FOLD, VAL_FOLD = 0, 1
SEED = 42
TEST_CREATED_FROM = "2022-01-01"

# Species used in the Milestone 1 report (27 species with >= 200 CC-licensed observations in India).
M1_CORE27_TAXON_IDS = {966799, 28868, 53492, 1304708, 29832, 30687, 1304706, 539844, 30264, 29574,
                       32150, 26654, 96224, 31079, 1161474, 1161478, 68530, 64091, 32205, 29550,
                       1161477, 1367260, 1499804, 27650, 29273, 1218631, 32227}

# WHO SEARO 2016 snakebite guidelines, India box (PDF pp. 86-87). Ophiophagus kaalinga was split
# from O. hannah in 2024, so it keeps the category of O. hannah.
WHO_CATEGORY_1 = {"Bungarus caeruleus", "Naja kaouthia", "Naja naja",
                  "Daboia russelii", "Echis carinatus", "Hypnale hypnale"}
WHO_CATEGORY_2 = {"Bungarus fasciatus", "Bungarus niger", "Bungarus sindanus", "Bungarus walli",
                  "Naja oxiana", "Naja sagittifera", "Ophiophagus hannah", "Ophiophagus kaalinga",
                  "Trimeresurus albolabris", "Trimeresurus erythrurus", "Trimeresurus purpureomaculatus",
                  "Craspedocephalus malabaricus", "Craspedocephalus gramineus", "Macrovipera lebetinus"}

In [ ]:
assert sklearn.__version__ == "1.9.1", f"restart the kernel after the pip install (found {sklearn.__version__})"
OWNER = kagglehub.whoami()["username"]
raw_root = Path(kagglehub.dataset_download(RAW_HANDLE))
print("running as", OWNER, "| raw dataset:", raw_root)

In [ ]:
def fetch(url: str) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, "Accept-Encoding": "gzip"})
    for attempt in range(RETRIES):
        try:
            with urllib.request.urlopen(request, timeout=TIMEOUT_S) as response:
                body = response.read()
                return gzip.decompress(body) if response.headers.get("Content-Encoding") == "gzip" else body
        except urllib.error.HTTPError as error:
            if error.code in (403, 404) or attempt == RETRIES - 1:
                raise
        except Exception:
            if attempt == RETRIES - 1:
                raise
        time.sleep(API_PAUSE_S * (attempt + 2))


def fetch_json(url: str) -> dict:
    return json.loads(fetch(url))


def slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

In [ ]:
def fetch_families(taxon_ids):
    families = {}
    for start in range(0, len(taxon_ids), TAXA_BATCH):
        batch = ",".join(str(t) for t in taxon_ids[start:start + TAXA_BATCH])
        for taxon in fetch_json(f"{INAT_API}/taxa/{batch}")["results"]:
            families[taxon["id"]] = next((a["name"] for a in taxon.get("ancestors", []) if a.get("rank") == "family"), None)
        time.sleep(API_PAUSE_S)
    return families


def who_category(species):
    return "1" if species in WHO_CATEGORY_1 else "2" if species in WHO_CATEGORY_2 else "not_listed"


def clean(df):
    df = df.copy()
    df["drop_reason"] = ""
    small = (df["bytes"] < MIN_BYTES) | (df[["width", "height"]].min(axis=1) < MIN_SIDE_PX)
    df.loc[small, "drop_reason"] = "too_small"
    species_per_hash = df.groupby("sha1")["species"].nunique()
    conflicting = set(species_per_hash[species_per_hash > 1].index)
    df.loc[df["sha1"].isin(conflicting) & (df["drop_reason"] == ""), "drop_reason"] = "duplicate_conflicting_species"
    order = df.sort_values(["in_india", "obs_id"], ascending=[False, True])
    repeat = set(order.index[order.duplicated("sha1", keep="first")])
    df.loc[df.index.isin(repeat) & (df["drop_reason"] == ""), "drop_reason"] = "exact_duplicate"
    return df[df["drop_reason"] == ""].drop(columns="drop_reason"), df[df["drop_reason"] != ""]


def assign_splits(df):
    folds = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_of = np.full(len(df), -1)
    for fold, (_, idx) in enumerate(folds.split(df, df["species"], groups=df["observer_id"])):
        fold_of[idx] = fold
    split = np.where(fold_of == TEST_FOLD, "test", np.where(fold_of == VAL_FOLD, "val", "train"))
    eligible = (df["in_india"] & (df["created_at"] >= TEST_CREATED_FROM)).to_numpy()
    return pd.Series(np.where((split == "test") & ~eligible, "test_extra", split), index=df.index)


raw = pd.read_csv(raw_root / "manifest.csv", dtype={"obs_id": str, "photo_id": str, "observer_id": str, "sha1": str})
raw["in_india"] = raw["in_india"].astype(str) == "True"
raw["created_at"] = raw["created_at"].fillna("").astype(str)
kept, dropped = clean(raw)
counts = kept["species"].value_counts()
classes = sorted(counts[counts >= MIN_PHOTOS_PER_CLASS].index)
data = kept[kept["species"].isin(classes)].copy()
unknown = kept[~kept["species"].isin(classes)].copy()
data["split"] = assign_splits(data)
unknown["split"] = "unknown_pool"

families = fetch_families(sorted({int(t) for t in raw["taxon_id"].dropna().unique()}))
splits = pd.concat([data, unknown])
splits["genus"] = splits["species"].str.split().str[0]
splits["family"] = splits["taxon_id"].map(lambda t: families.get(int(t)) if pd.notna(t) else None)
splits["who_category"] = splits["species"].map(who_category)
tier_counts = data["species"].value_counts()
splits["tier"] = [("unknown_pool" if s == "unknown_pool" else "tier1" if tier_counts[sp] >= TIER1_MIN_PHOTOS else "tier2")
                  for s, sp in zip(splits["split"], splits["species"])]
splits["m1_core27"] = splits["taxon_id"].map(lambda t: pd.notna(t) and int(t) in M1_CORE27_TAXON_IDS)
splits["cc_licensed"] = splits["license"] != "all-rights-reserved"
splits["path"] = [f"{s}/{slug(sp)}/{Path(f).name}" for s, sp, f in zip(splits["split"], splits["species"], splits["file"])]
splits.to_csv(PREP_DIR / "splits.csv", index=False)
dropped.to_csv(PREP_DIR / "excluded.csv", index=False)

In [ ]:
labelled_counts = splits[splits["split"] != "unknown_pool"].pivot_table(
    index="species", columns="split", values="photo_id", aggfunc="count", fill_value=0)
labelled_counts = labelled_counts.reindex(columns=["train", "val", "test", "test_extra"], fill_value=0)
meta_cols = ["genus", "family", "who_category", "tier", "m1_core27", "taxon_id"]
classes_out = labelled_counts.join(splits.drop_duplicates("species").set_index("species")[meta_cols])
classes_out["total"] = classes_out[["train", "val", "test", "test_extra"]].sum(axis=1)
classes_out = classes_out.sort_values("total", ascending=False)
classes_out.insert(0, "class_id", range(len(classes_out)))
classes_out.to_csv(PREP_DIR / "classes.csv")
report = {"photos": len(splits), "classes": len(classes_out), "dropped": dropped["drop_reason"].value_counts().to_dict(),
          "split_photos": splits["split"].value_counts().to_dict(), "scikit_learn": sklearn.__version__}
(PREP_DIR / "split_report.json").write_text(json.dumps(report, indent=1, default=str))

published = pd.read_csv(Path(kagglehub.dataset_download(SPLITS_HANDLE)) / "splits.csv", dtype={"photo_id": str})
same = splits.set_index("path")["split"].reindex(published["path"]).eq(published["split"].values).mean()
print(f"agrees with the team's published split for {same:.2%} of photos")

if UPLOAD:
    (PREP_DIR / "dataset-metadata.json").unlink(missing_ok=True)
    kagglehub.dataset_upload(f"{OWNER}/serpentguard-m2-splits", str(PREP_DIR), version_notes="02_clean_and_split")